### Mapping the Seasonality intensity with Flown Data

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.gridspec as gridspec
import numpy as np
from scipy.ndimage import gaussian_filter1d
from scipy.signal import savgol_filter
import fsspec

# Loading the excel Flown data and copying it
flown_data = pd.read_excel("Final_Data.xlsx",sheet_name=None)
flown_data = flown_data['Sheet1'].copy()

In [115]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.gridspec as gridspec
import numpy as np
from scipy.ndimage import gaussian_filter1d
from scipy.signal import savgol_filter
import fsspec

# Loading the excel Flown data and copying it
flown_data = pd.read_excel("Final_Data.xlsx",sheet_name=None)
flown_data = flown_data['Sheet1'].copy()

# Copying the data
data = flown_data.copy()
# Renaming the column names 
data.rename(columns={'DI Level': 'TI Level','DI Toggle': 'TI Toggle','WT_DI': 'WT_TI'},inplace=True)

# 2. Define columns with the EXACT casing from your dataset
required_columns = ['corridor', 'AirCraftType', 'TI Level', 'LI Level','flightEstimatedDepartureTime_ET', 
                    'TI Toggle', 'LI Toggle', 'WT_TI', 'WT_LI', 'Final Toggle', 'Toggle Delta']

# 3. Create the filtered dataframe
req_data = data.loc[:,required_columns].copy()

# 4. getting the month from the date table
req_data['Month'] = req_data['flightEstimatedDepartureTime_ET'].dt.month

# 5. Droping the column of date that we dont need
req_data.drop(columns=['flightEstimatedDepartureTime_ET'],inplace=True)

# 6. Loading the Seasonality data
si_data = pd.read_csv("lj_smid_kmean_intensity_slab_va.csv")

# 7. mapping the month names to number
month_mapping = {
    'Jan': 1, 'Feb': 2, 'Mar': 3, 'Apr': 4, 
    'May': 5, 'Jun': 6, 'Jul': 7, 'Aug': 8, 
    'Sep': 9, 'Oct': 10, 'Nov': 11, 'Dec': 12
}
si_data['month'] = si_data['month'].map(month_mapping)

# 8. Removed space from the corridor name
si_data['corridor'] = si_data['corridor'].str.replace(' -> ', '->', regex=False)

# 9. Rename to match req_data column name
si_data.rename(columns={'AircraftType': 'AirCraftType'}, inplace=True)

# 10. Standardize AircraftType values to match req_data
si_data['AirCraftType'] = si_data['AirCraftType'].map({
                        'Light Jet'        : 'Light',
                        'Super Midsize Jet': 'Super Midsize',
                        'Super Midsize'    : 'Super Midsize',
                        'Light'            : 'Light'})

# 11. correcting the month name
req_data.rename(columns={'Month':'month'},inplace=True)

# 12. Merging
si_data_to_join = si_data[['AirCraftType', 'corridor', 'month', 
                                   'density_pct', 'cluster_label', 
                                   'slab', 'intensity']].copy()

# Deduplicate si_data on join keys — safety check
si_data_to_join = si_data_to_join.drop_duplicates(
    subset=['AirCraftType', 'corridor', 'month'], keep='last').reset_index(drop=True)

# ============================================================
# STEP 5: Left Join req_data with si_data on (AirCraftType, corridor, month)
# ============================================================
final_result = pd.merge(
    req_data,
    si_data_to_join,
    on=['AirCraftType', 'corridor', 'month'],
    how='left')

# ============================================================
# STEP 6: Validate
# ============================================================
print(f"req_data rows    : {len(req_data)}")
print(f"final_result rows: {len(final_result)}")
assert len(final_result) == len(req_data), "⚠️ Row count mismatch — duplicates in si_data!"
print("✅ Row count matches!")

# 15. Updating the weights of TI and LI and SI
final_result['WT_TI'] = 0.6
final_result['WT_LI'] = 0.1
final_result['WT_SI'] = 0.3


# Renamin intensity to SI_Toggle
final_result.rename(columns={'intensity':'SI_Toggle'},inplace=True)

import numpy as np

# 1. Helper function to calculate the mean of a range string (e.g., "3 - 4" -> 3.5)
def get_range_mean(range_val):
    if pd.isna(range_val) or not isinstance(range_val, str):
        return np.nan
    try:
        # Splits "3 - 4" into [3.0, 4.0] and returns the average
        parts = [float(x.strip()) for x in range_val.split('-')]
        return sum(parts) / len(parts)
    except:
        return np.nan

# 2. Apply the formula row-wise to create final_intensity
# Formula: WT_TI * mean(TI Level) + WT_LI * mean(LI Level) + WT_SI * SI_Toggle
final_result['final_intensity'] = (
    final_result['WT_TI'] * final_result['TI Level'].apply(get_range_mean) +
    final_result['WT_LI'] * final_result['LI Level'].apply(get_range_mean) +
    final_result['WT_SI'] * final_result['SI_Toggle']
)

# 3. View the results (including your example case)
final_result[['corridor', 'TI Level', 'LI Level', 'SI_Toggle', 'final_intensity']].head()

# 1. Define the conditions based on final_intensity (CTS)
conditions = [
    (final_result['final_intensity'] < 2),
    (final_result['final_intensity'] >= 2) & (final_result['final_intensity'] < 3),
    (final_result['final_intensity'] >= 3) & (final_result['final_intensity'] < 4),
    (final_result['final_intensity'] >= 4) & (final_result['final_intensity'] < 5),
    (final_result['final_intensity'] >= 5) & (final_result['final_intensity'] < 6),
    (final_result['final_intensity'] >= 6) & (final_result['final_intensity'] < 7),
    (final_result['final_intensity'] >= 7) & (final_result['final_intensity'] < 8),
    (final_result['final_intensity'] >= 8)
]

# 2. Define the corresponding toggle percentages
choices = [
    0.00,  # < 2
    0.10,  # 2 TO < 3
    0.15,  # 3 TO < 4
    0.20,  # 4 TO < 5
    0.25,  # 5 TO < 6
    0.30,  # 6 TO < 7
    0.35,  # 7 TO < 8
    0.40   # 8+
]

# 3. Update the 'Final Toggle' column
# We use np.select to apply the logic; default to NaN if intensity is missing
final_result['Final Toggle'] = np.select(conditions, choices, default=np.nan)


req_data rows    : 115071
final_result rows: 115071
✅ Row count matches!


,corridor,AirCraftType,TI Level,LI Level,TI Toggle,LI Toggle,WT_TI,WT_LI,Final Toggle,Toggle Delta,month,density_pct,cluster_label,slab,SI_Toggle,WT_SI,final_intensity
0,South Florida->New York City,Super Midsize,3 - 4,1 - 2,0.15,0.05,0.6,0.1,0.15,-1818.624,4,12.29,winter-spring-peak,12%-15%,4.0,0.3,3.45


In [116]:
# 1. Define columns to drop from original 'data' to avoid stale/duplicate values
# We want the NEW versions of these from final_result
cols_to_overwrite = ['Final Toggle', 'WT_TI', 'WT_LI', 'WT_SI']
data_clean = data.drop(columns=cols_to_overwrite, errors='ignore')

# 2. Identify new columns in final_result that aren't in our cleaned data
# This will include the new SI_Toggle, final_intensity, and the updated weights/toggles
new_cols_to_add = [col for col in final_result.columns if col not in data_clean.columns]

# 3. Merge safely
# Using a merge/join is safer than positional concat, 
# but if you are certain of order, we'll keep it fast:
final_merged = pd.concat([
    data_clean.reset_index(drop=True), 
    final_result[new_cols_to_add].reset_index(drop=True)
], axis=1)

# 4. Validation (Concise)
if len(final_merged) == len(data):
    print(f"✅ Success: {len(final_merged)} rows processed.")
    print(f"Total columns: {len(final_merged.columns)}")
else:
    print("⚠️ Warning: Row count mismatch!")

# 5. Final Display
sample_cols = [
    'corridor', 'AirCraftType', 'month', 'SI_Toggle', 
    'WT_TI', 'WT_LI', 'WT_SI', 'final_intensity', 'Final Toggle'
]
existing_sample = [c for c in sample_cols if c in final_merged.columns]
final_merged[existing_sample].head(1)

✅ Success: 115071 rows processed.
Total columns: 67


,corridor,AirCraftType,month,SI_Toggle,WT_TI,WT_LI,WT_SI,final_intensity,Final Toggle
0,South Florida->New York City,Super Midsize,4,4.0,0.6,0.1,0.3,3.45,0.15


In [121]:
# 1. Create a mask for rows where SI_Toggle is NULL
si_null_mask = final_merged['SI_Toggle'].isna()

# 2. Update the weights for these specific rows
final_merged.loc[si_null_mask, 'WT_TI'] = 0.8
final_merged.loc[si_null_mask, 'WT_LI'] = 0.2
final_merged.loc[si_null_mask, 'WT_SI'] = 0

# 3. Update the final_intensity for these rows
# Since WT_SI is 0 for these rows, the SI component is effectively ignored
final_merged.loc[si_null_mask, 'final_intensity'] = (
    final_merged.loc[si_null_mask, 'WT_TI'] * final_merged.loc[si_null_mask, 'TI Level'].apply(get_range_mean) +
    final_merged.loc[si_null_mask, 'WT_LI'] * final_merged.loc[si_null_mask, 'LI Level'].apply(get_range_mean)
)

# 4. (Optional but Recommended) Update 'Final Toggle' and 'Toggle Delta' 
# Since final_intensity has changed, these columns should be refreshed for consistency
final_merged['Final Toggle'] = np.select(conditions, choices, default=np.nan)

final_merged['Toggle Delta'] = (
    final_merged['flightcost'] / (1 + final_merged['final_adjustment_pct'] / 100)
) * (final_merged['Final Toggle'] - final_merged['final_adjustment_pct'] / 100)

# Verify the changes for a few rows that were previously NULL
final_merged[si_null_mask][['corridor', 'WT_TI', 'WT_LI', 'WT_SI', 'final_intensity', 'Final Toggle']].head()

,corridor,WT_TI,WT_LI,WT_SI,final_intensity,Final Toggle
2,San Antonio->Dallas,0.8,0.2,0.0,2.9,NaN
7,North Florida->North Florida,0.8,0.2,0.0,NaN,NaN
9,Philadelphia->Charlotte,0.8,0.2,0.0,2.9,NaN
16,DMV->New York City,0.8,0.2,0.0,3.3,NaN
29,LA Basin->LA Basin,0.8,0.2,0.0,NaN,NaN


In [ ]:
# Define the desired column order based on your list
ordered_columns = [
    'flightId', 'atlasreservationid', 'flightStatus', 'legOrder', 
    'flightoriginAirportId', 'flightoriginAirport', 'flightoriginAirportName', 
    'flightoriginAirportCity', 'flightoriginAirportState', 'flightoriginAirportCountry', 
    'flightdestinationAirportId', 'flightdestinationAirport', 'flightdestinationAirportName', 
    'flightdestinationAirportCity', 'flightdestinationAirportState', 'flightdestinationAirportCountry', 
    'flightEstimatedDepartureTime', 'flightEstimatedArrivalTime', 'flightEstimatedFlightHours', 
    'flightEstimatedBilledHours', 'flightActualDepartureTime', 'flightActualArrivalTime', 
    'flightActualFlightHours', 'flightActualBilledHours', 'flightrequestedAircraftType', 
    'flightrequestedAircraftCabinId', 'flightrequestedAircraftCabinName', 'flightrequestedAircraftTypeId', 
    'flightrequestedAircraftTypeName', 'flighttailNumber', 'flightactualAircraftCabinId', 
    'AirCraftType', 'flightactualAircraftId', 'flightactualAircraftTypeName', 'operatorName', 
    'operatorType', 'flightcost', 'reservationId', 'reservationStatus', 'reservationCreateDate', 
    'bookerMemberId', 'bookerContactId', 'passengerId', 'from_metro', 'to_metro', 'corridor', 
    'DOW', 'TOD', 'flightoriginAirportRegion', 'flightdestinationAirportRegion', 
    'flightEstimatedDepartureTime_ET', 'month', 'final_adjustment_pct', 'TI Level', 
    'LI Level', 'TI Toggle', 'LI Toggle', 'SI_Toggle', 'Toggle Delta', 'WT_TI', 
    'WT_LI', 'WT_SI', 'density_pct', 'cluster_label', 'slab', 'Final Toggle', 'final_intensity'
]

# Filter ordered_columns to only include those actually present in final_merged
available_columns = [col for col in ordered_columns if col in final_merged.columns]

# Reorder the dataframe
final_merged = final_merged[available_columns]

# Verification: Display the first few rows with the new order
final_merged.head(1)

In [2]:
import pandas as pd
import numpy as np

# ============================================================
# STEP 1: Load and prepare flown data
# ============================================================
# flown_data = pd.read_excel("Final_Data.xlsx", sheet_name=None)
data = flown_data.copy()

# Rename DI columns to TI equivalents
data.rename(columns={
    'DI Level': 'TI Level',
    'DI Toggle': 'TI Toggle',
    'WT_DI': 'WT_TI'
}, inplace=True)

# ============================================================
# STEP 2: Extract required columns and derive month
# ============================================================
required_columns = [
    'corridor', 'AirCraftType', 'TI Level', 'LI Level',
    'flightEstimatedDepartureTime_ET',
    'TI Toggle', 'LI Toggle', 'WT_TI', 'WT_LI', 'Final Toggle', 'Toggle Delta'
]

req_data = data.loc[:, required_columns].copy()
req_data['month'] = req_data['flightEstimatedDepartureTime_ET'].dt.month
req_data.drop(columns=['flightEstimatedDepartureTime_ET'], inplace=True)

# ============================================================
# STEP 3: Load and prepare seasonality data
# ============================================================
si_data = pd.read_csv("lj_smid_kmean_intensity_slab_va.csv")

month_mapping = {
    'Jan': 1, 'Feb': 2, 'Mar': 3, 'Apr': 4,
    'May': 5, 'Jun': 6, 'Jul': 7, 'Aug': 8,
    'Sep': 9, 'Oct': 10, 'Nov': 11, 'Dec': 12
}
si_data['month'] = si_data['month'].map(month_mapping)

# Clean corridor formatting
si_data['corridor'] = si_data['corridor'].str.replace(' -> ', '->', regex=False)

# Rename to match req_data
si_data.rename(columns={'AircraftType': 'AirCraftType'}, inplace=True)

# Standardize AircraftType values
si_data['AirCraftType'] = si_data['AirCraftType'].map({
    'Light Jet':         'Light',
    'Super Midsize Jet': 'Super Midsize',
    'Super Midsize':     'Super Midsize',
    'Light':             'Light'
})

# ============================================================
# STEP 4: Merge req_data with seasonality data
# ============================================================
si_data_to_join = si_data[[
    'AirCraftType', 'corridor', 'month',
    'density_pct', 'cluster_label', 'slab', 'intensity'
]].copy()

# Deduplicate on join keys
si_data_to_join = si_data_to_join.drop_duplicates(
    subset=['AirCraftType', 'corridor', 'month'], keep='last'
).reset_index(drop=True)

final_result = pd.merge(
    req_data,
    si_data_to_join,
    on=['AirCraftType', 'corridor', 'month'],
    how='left'
)

# Validate row count
print(f"req_data rows    : {len(req_data)}")
print(f"final_result rows: {len(final_result)}")
assert len(final_result) == len(req_data), "⚠️ Row count mismatch — duplicates in si_data!"
print("✅ Row count matches!")

# ============================================================
# STEP 5: Set base weights and rename intensity
# ============================================================
final_result['WT_TI'] = 0.6
final_result['WT_LI'] = 0.1
final_result['WT_SI'] = 0.3

final_result.rename(columns={'intensity': 'SI_Toggle'}, inplace=True)

# ============================================================
# STEP 6: Compute final_intensity using range means
# ============================================================
def get_range_mean(range_val):
    """Convert '3 - 4' to 3.5, and handle '10+' by returning 10."""
    if pd.isna(range_val) or not isinstance(range_val, str):
        return np.nan
    try:
        # Handle "10+" case: remove plus and convert to float
        if '+' in range_val:
            return float(range_val.replace('+', '').strip())
        
        # Handle "3 - 4" case
        parts = [float(x.strip()) for x in range_val.split('-')]
        return sum(parts) / len(parts)
    except Exception:
        return np.nan


final_result['final_intensity'] = (
    final_result['WT_TI'] * final_result['TI Level'].apply(get_range_mean) +
    final_result['WT_LI'] * final_result['LI Level'].apply(get_range_mean) +
    final_result['WT_SI'] * final_result['SI_Toggle']
)

# ============================================================
# STEP 7: Helper — build toggle conditions & choices
# ============================================================
def build_toggle_conditions(df):
    """Return (conditions, choices) lists based on final_intensity column."""
    fi = df['final_intensity']
    conditions = [
        fi < 2,
        (fi >= 2) & (fi < 3),
        (fi >= 3) & (fi < 4),
        (fi >= 4) & (fi < 5),
        (fi >= 5) & (fi < 6),
        (fi >= 6) & (fi < 7),
        (fi >= 7) & (fi < 8),
        fi >= 8
    ]
    choices = [0.00, 0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40]
    return conditions, choices


conditions, choices = build_toggle_conditions(final_result)
final_result['Final Toggle'] = np.select(conditions, choices, default=np.nan)

# ============================================================
# STEP 8: Merge back into original data
# ============================================================
cols_to_overwrite = ['Final Toggle', 'WT_TI', 'WT_LI', 'WT_SI']
data_clean = data.drop(columns=cols_to_overwrite, errors='ignore')

new_cols_to_add = [col for col in final_result.columns if col not in data_clean.columns]

final_merged = pd.concat([
    data_clean.reset_index(drop=True),
    final_result[new_cols_to_add].reset_index(drop=True)
], axis=1)

if len(final_merged) == len(data):
    print(f"✅ Success: {len(final_merged)} rows processed.")
    print(f"Total columns: {len(final_merged.columns)}")
else:
    print("⚠️ Warning: Row count mismatch!")

# ============================================================
# STEP 9: Handle rows where SI_Toggle is NULL (adjust weights)
# ============================================================
si_null_mask = final_merged['SI_Toggle'].isna()

final_merged.loc[si_null_mask, 'WT_TI'] = 0.8
final_merged.loc[si_null_mask, 'WT_LI'] = 0.2
final_merged.loc[si_null_mask, 'WT_SI'] = 0.0

final_merged.loc[si_null_mask, 'final_intensity'] = (
    final_merged.loc[si_null_mask, 'WT_TI'] * final_merged.loc[si_null_mask, 'TI Level'].apply(get_range_mean) +
    final_merged.loc[si_null_mask, 'WT_LI'] * final_merged.loc[si_null_mask, 'LI Level'].apply(get_range_mean)
)

# ============================================================
# STEP 10: Recompute Final Toggle and Toggle Delta
# ============================================================
conditions, choices = build_toggle_conditions(final_merged)
final_merged['Final Toggle'] = np.select(conditions, choices, default=np.nan)

# Compute Toggle Delta only if required columns exist
if 'flightcost' in final_merged.columns and 'final_adjustment_pct' in final_merged.columns:
    final_merged['Toggle Delta'] = (
        final_merged['flightcost'] / (1 + final_merged['final_adjustment_pct'] / 100)
    ) * (final_merged['Final Toggle'] - final_merged['final_adjustment_pct'] / 100)
else:
    print("⚠️ Skipping Toggle Delta: 'flightcost' or 'final_adjustment_pct' not found in data.")

# ============================================================
# STEP 11: Reorder columns
# ============================================================
ordered_columns = [
    'flightId', 'atlasreservationid', 'flightStatus', 'legOrder',
    'flightoriginAirportId', 'flightoriginAirport', 'flightoriginAirportName',
    'flightoriginAirportCity', 'flightoriginAirportState', 'flightoriginAirportCountry',
    'flightdestinationAirportId', 'flightdestinationAirport', 'flightdestinationAirportName',
    'flightdestinationAirportCity', 'flightdestinationAirportState', 'flightdestinationAirportCountry',
    'flightEstimatedDepartureTime', 'flightEstimatedArrivalTime', 'flightEstimatedFlightHours',
    'flightEstimatedBilledHours', 'flightActualDepartureTime', 'flightActualArrivalTime',
    'flightActualFlightHours', 'flightActualBilledHours', 'flightrequestedAircraftType',
    'flightrequestedAircraftCabinId', 'flightrequestedAircraftCabinName', 'flightrequestedAircraftTypeId',
    'flightrequestedAircraftTypeName', 'flighttailNumber', 'flightactualAircraftCabinId',
    'AirCraftType', 'flightactualAircraftId', 'flightactualAircraftTypeName', 'operatorName',
    'operatorType', 'flightcost', 'reservationId', 'reservationStatus', 'reservationCreateDate',
    'bookerMemberId', 'bookerContactId', 'passengerId', 'from_metro', 'to_metro', 'corridor',
    'DOW', 'TOD', 'flightoriginAirportRegion', 'flightdestinationAirportRegion',
    'flightEstimatedDepartureTime_ET', 'month', 'final_adjustment_pct', 'TI Level',
    'LI Level', 'TI Toggle', 'LI Toggle', 'SI_Toggle', 'Toggle Delta', 'WT_TI',
    'WT_LI', 'WT_SI', 'density_pct', 'cluster_label', 'slab', 'Final Toggle', 'final_intensity'
]

available_columns = [col for col in ordered_columns if col in final_merged.columns]
final_merged = final_merged[available_columns]

print(f"\n✅ Pipeline complete. Final shape: {final_merged.shape}")
print(final_merged.head())

req_data rows    : 115071
final_result rows: 115071
✅ Row count matches!
✅ Success: 115071 rows processed.
Total columns: 67

✅ Pipeline complete. Final shape: (115071, 67)
   flightId  atlasreservationid flightStatus  legOrder  flightoriginAirportId  \
0   1208330              794521        Flown         1                    333   
1   1208345              794531        Flown         1                    810   
2   1208363              794541        Flown         1                     93   
3   1208371              794548        Flown         1                    160   
4   1208372              794548        Flown         2                   1026   

  flightoriginAirport         flightoriginAirportName flightoriginAirportCity  \
0                KPBI        Palm Beach International         West Palm Beach   
1                KTEB                       Teterboro               Teterboro   
2                KAUS  Austin Bergstrom International                  Austin   
3               

In [9]:
import time
st = time.time()
final_merged.to_csv("Seasonal_Intensity_4.csv")
et = time.time()
print(et-st)

7.040568828582764


In [13]:
import time
st = time.time()
final_merged.to_excel("Seasonal_Intensity_4.xlsx",index=False)
et = time.time()
print(et-st)

227.08329606056213


In [3]:
final_merged['Final Toggle'].isna().sum()

np.int64(6216)

# Seasonality correct weights finder
# -------------------------------------------------------------

In [20]:
# Splitting the file into two jets
light_jet = final_merged[final_merged['AirCraftType']=='Light']
SMID = final_merged[final_merged['AirCraftType']=='Super Midsize']


In [39]:
try:
    # 1. Clean the data
    # Remove rows where adjustment is -100 (division by zero)
    # Remove rows where flightcost or final_adjustment_pct is missing
    # Remove rows where adjustment is close to -100 (near-zero division)
    final_merged = final_merged[
        (final_merged['final_adjustment_pct'] != -100) &
        (final_merged['flightcost'].notna()) &
        (final_merged['final_adjustment_pct'].notna()) &
        (abs(1 + final_merged['final_adjustment_pct'] / 100) > 0.01)  # Threshold guard
    ].copy()

    # 2. Calculate the divisor once (reused across formulas)
    divisor = 1 + final_merged['final_adjustment_pct'] / 100

    # 3. Calculate Baseline cost
    # Formula: [flightcost / (1 + final_adjustment_pct/100)] * 0.8
    final_merged['Baseline cost'] = (final_merged['flightcost'] / divisor) * 0.8

    # 4. Calculate Original Toggle Amount
    # Formula: [flightcost - (flightcost / (1 + final_adjustment_pct/100))] * 0.8
    final_merged['Original Toggle Amount'] = (
        final_merged['flightcost'] - (final_merged['flightcost'] / divisor)
    ) * 0.8

    # 5. Calculate Toggle as per Composite Index (CI)
    # Formula: Baseline cost * Final Toggle
    if 'Final Toggle' not in final_merged.columns:
        raise KeyError("'Final Toggle' column is missing from the dataframe.")
    
    final_merged['Toggle as per CI'] = final_merged['Baseline cost'] * final_merged['Final Toggle']

    print("✅ New columns added successfully.")
    print(f"   Rows remaining after cleaning: {len(final_merged)}")
    print(
        final_merged[[
            'flightcost', 'final_adjustment_pct',
            'Baseline cost', 'Original Toggle Amount', 'Toggle as per CI'
        ]].head()
    )

except KeyError as e:
    print(f"❌ Missing column error: {e}")
except ZeroDivisionError as e:
    print(f"❌ Division by zero error: {e}")
except Exception as e:
    print(f"❌ Unexpected error during calculation: {e}")

✅ New columns added successfully.
   Rows remaining after cleaning: 114867
   flightcost  final_adjustment_pct  Baseline cost  Original Toggle Amount  \
0     18944.0                    25       12124.16                 3031.04   
1     22724.0                    15       15808.00                 2371.20   
2       495.0                     0         396.00                    0.00   
3     15840.0                     0       12672.00                    0.00   
4     16560.0                    25       10598.40                 2649.60   

   Toggle as per CI  
0          1818.624  
1          1580.800  
2            39.600  
3          1900.800  
4          2119.680  


In [1]:
import pandas as pd
final_merged = pd.read_excel("Seasonal_Intensity_4.xlsx")
final_merged.head()

,flightId,atlasreservationid,flightStatus,legOrder,flightoriginAirportId,flightoriginAirport,flightoriginAirportName,flightoriginAirportCity,flightoriginAirportState,flightoriginAirportCountry,...,SI_Toggle,Toggle Delta,WT_TI,WT_LI,WT_SI,density_pct,cluster_label,slab,Final Toggle,final_intensity
0,1208330,794521,Flown,1,333,KPBI,Palm Beach International,West Palm Beach,FL,US,...,4.0,-1515.52,0.6,0.1,0.3,12.29,winter-spring-peak,12%-15%,0.15,3.45
1,1208345,794531,Flown,1,810,KTEB,Teterboro,Teterboro,NJ,US,...,2.0,-988.00,0.6,0.1,0.3,8.93,winter-spring-peak,6%-9%,0.10,2.85
2,1208363,794541,Flown,1,93,KAUS,Austin Bergstrom International,Austin,TX,US,...,NaN,49.50,0.8,0.2,0.0,NaN,NaN,NaN,0.10,2.90
3,1208371,794548,Flown,1,160,KBNA,Nashville International,Nashville,TN,US,...,3.0,2376.00,0.6,0.1,0.3,9.97,winter-spring-peak,9%-12%,0.15,3.25
4,1208372,794548,Flown,2,1026,KBCT,Boca Raton,Boca Raton,FL,US,...,3.0,-662.40,0.6,0.1,0.3,10.48,winter-spring-peak,9%-12%,0.20,4.35


In [3]:
import pandas as pd
import numpy as np
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

# ============================================================
# HELPER: get_range_mean
# ============================================================
def get_range_mean(range_val):
    if pd.isna(range_val) or not isinstance(range_val, str):
        return np.nan
    try:
        if '+' in range_val:
            return float(range_val.replace('+', '').strip())
        parts = [float(x.strip()) for x in range_val.split('-')]
        return sum(parts) / len(parts)
    except Exception:
        return np.nan

# ============================================================
# HELPER: build_toggle_conditions
# ============================================================
def build_toggle_conditions(df):
    fi = df['final_intensity']
    conditions = [
        fi < 2,
        (fi >= 2) & (fi < 3),
        (fi >= 3) & (fi < 4),
        (fi >= 4) & (fi < 5),
        (fi >= 5) & (fi < 6),
        (fi >= 6) & (fi < 7),
        (fi >= 7) & (fi < 8),
        fi >= 8
    ]
    choices = [0.00, 0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40]
    return conditions, choices

# ============================================================
# CORE: Apply weights and compute columns
# ============================================================
def apply_weights_and_calculate(df, seasonal_weights, non_seasonal_weights):
    df = df.copy()

    seasonal_mask     = df['SI_Toggle'].notna() & (df['SI_Toggle'] != 0)
    non_seasonal_mask = ~seasonal_mask
    bins = [0, 2, 3, 4, 5, 6, 10]

    def assign_tier_weights(mask, weight_list):
        if mask.sum() == 0:
            return
        tier_indices = np.digitize(df.loc[mask, 'final_intensity'], bins) - 1
        tier_indices = np.clip(5 - tier_indices, 0, 5)

        # ✅ Explicitly map each weight key to correct column
        df.loc[mask, 'WT_TI'] = [weight_list[i]['TI'] for i in tier_indices]
        df.loc[mask, 'WT_SI'] = [weight_list[i]['SI'] for i in tier_indices]  # ← SI to WT_SI
        df.loc[mask, 'WT_LI'] = [weight_list[i]['LI'] for i in tier_indices]  # ← LI to WT_LI

    assign_tier_weights(seasonal_mask,     seasonal_weights)
    assign_tier_weights(non_seasonal_mask, non_seasonal_weights)

    df['final_intensity'] = (
        df['WT_TI'] * df['TI Level'].apply(get_range_mean) +
        df['WT_LI'] * df['LI Level'].apply(get_range_mean) +
        df['WT_SI'] * df['SI_Toggle'].fillna(0)
    )

    conditions, choices = build_toggle_conditions(df)
    df['Final Toggle'] = np.select(conditions, choices, default=np.nan)

    df = df[
        (df['final_adjustment_pct'] != -100) &
        (df['flightcost'].notna()) &
        (df['final_adjustment_pct'].notna()) &
        (abs(1 + df['final_adjustment_pct'] / 100) > 0.01)
    ].copy()

    divisor = 1 + df['final_adjustment_pct'] / 100
    df['Baseline cost']          = (df['flightcost'] / divisor) * 0.8
    df['Original Toggle Amount'] = (df['flightcost'] - df['flightcost'] / divisor) * 0.8
    df['Toggle as per CI']       = df['Baseline cost'] * df['Final Toggle']

    return df

# ============================================================
# SUMMARY: Compute metrics
# ============================================================
def compute_summary(df):
    original_toggle_m = df['Original Toggle Amount'].sum() / 1_000_000
    ci_toggle_m       = df['Toggle as per CI'].sum()       / 1_000_000
    non_null_ci       = df[df['Toggle as per CI'].notna()]
    count_ci_less     = (non_null_ci['Toggle as per CI'] < non_null_ci['Original Toggle Amount']).sum()
    return round(original_toggle_m, 2), round(ci_toggle_m, 2), int(count_ci_less)

# ============================================================
# BUILD SUMMARY BLOCK ROWS
# ============================================================
def build_summary_block(combo_label, seasonal_weights, non_seasonal_weights,
                         orig_toggle, ci_toggle, count_ci_less):
    rows = []
    rows.append({
        'Block': combo_label,
        'S_TI': 'TI', 'S_SI': 'SI', 'S_LI': 'LI',
        'NS_TI': 'TI', 'NS_SI': 'SI', 'NS_LI': 'LI',
        'Original Toggle (m $)': 'Original Toggle (m $)',
        'CI based toggle (m $)': 'CI based toggle (m $)',
        'Count (CI toggle < WU toggle)': 'Count (CI toggle % < WU toggle %) for non NULL CI only'
    })
    for i in range(6):
        sw  = seasonal_weights[i]
        nsw = non_seasonal_weights[i]
        rows.append({
            'Block': '',
            'S_TI': sw['TI'],   'S_SI': sw['SI'],   'S_LI': sw['LI'],
            'NS_TI': nsw['TI'], 'NS_SI': nsw['SI'], 'NS_LI': nsw['LI'],
            'Original Toggle (m $)': orig_toggle   if i == 0 else '',
            'CI based toggle (m $)': ci_toggle     if i == 0 else '',
            'Count (CI toggle < WU toggle)': count_ci_less if i == 0 else ''
        })
    rows.append({k: '' for k in rows[0].keys()})
    rows.append({k: '' for k in rows[0].keys()})
    return rows

# ============================================================
# STORAGE LISTS — accumulate across cells
# ============================================================
all_summary_rows  = []   # list of row dicts → becomes summary sheet
all_detail_frames = {}   # dict: { combo_label: df } → one sheet each

print("✅ Helpers and storage initialized.")

✅ Helpers and storage initialized.


In [4]:
# ============================================================
# DEFINE & RUN — Combination 1
# ============================================================
combo_1 = {
    'label': 'Combination 1',
    'seasonal': [
        {'TI': 0.6, 'SI': 0.3, 'LI': 0.1},
        {'TI': 0.7, 'SI': 0.2, 'LI': 0.1},
        {'TI': 0.8, 'SI': 0.2, 'LI': 0.0},
        {'TI': 0.8, 'SI': 0.2, 'LI': 0.0},
        {'TI': 0.8, 'SI': 0.2, 'LI': 0.0},
        {'TI': 0.5, 'SI': 0.5, 'LI': 0.0},
    ],
    'non_seasonal': [
        {'TI': 0.8, 'LI': 0.2, 'SI': 0.0},
        {'TI': 0.8, 'LI': 0.2, 'SI': 0.0},
        {'TI': 0.7, 'LI': 0.3, 'SI': 0.0},
        {'TI': 0.7, 'LI': 0.3, 'SI': 0.0},
        {'TI': 0.6, 'LI': 0.4, 'SI': 0.0},
        {'TI': 0.6, 'LI': 0.4, 'SI': 0.0},
    ]
}

result_df_1 = apply_weights_and_calculate(
    final_merged, combo_1['seasonal'], combo_1['non_seasonal']
)
orig_1, ci_1, count_1 = compute_summary(result_df_1)
print(f"Combination 1 → Original: {orig_1} m$  |  CI: {ci_1} m$  |  Count: {count_1}")

# Store summary rows
all_summary_rows.extend(
    build_summary_block(combo_1['label'], combo_1['seasonal'],
                        combo_1['non_seasonal'], orig_1, ci_1, count_1)
)

# Store detail — only keep essential columns to save memory
all_detail_frames['Combination 1'] = result_df_1[[
    'flightcost', 'final_adjustment_pct', 'Final Toggle',
    'Baseline cost', 'Original Toggle Amount', 'Toggle as per CI',
    'WT_TI', 'WT_LI', 'WT_SI', 'final_intensity',
    'TI Level', 'LI Level', 'SI_Toggle'
]].copy()

# Free intermediate result from memory
del result_df_1
print("✅ Combination 1 stored. Intermediate result cleared from memory.")

Combination 1 → Original: 185.03 m$  |  CI: 188.56 m$  |  Count: 45124
✅ Combination 1 stored. Intermediate result cleared from memory.


In [5]:
# ============================================================
# DEFINE & RUN — Combination 2  (edit weights as needed)
# ============================================================
combo_2 = {
    'label': 'Combination 2',
    'seasonal': [
        {'TI': 0.6, 'SI': 0.3, 'LI': 0.1},
        {'TI': 0.7, 'SI': 0.2, 'LI': 0.1},
        {'TI': 0.8, 'SI': 0.2, 'LI': 0.0},
        {'TI': 0.8, 'SI': 0.2, 'LI': 0.0},
        {'TI': 0.8, 'SI': 0.2, 'LI': 0.0},
        {'TI': 0.5, 'SI': 0.5, 'LI': 0.0},
    ],
    'non_seasonal': [
        {'TI': 0.8, 'LI': 0.2, 'SI': 0.0},
        {'TI': 0.8, 'LI': 0.2, 'SI': 0.0},
        {'TI': 0.7, 'LI': 0.3, 'SI': 0.0},
        {'TI': 0.7, 'LI': 0.3, 'SI': 0.0},
        {'TI': 0.6, 'LI': 0.4, 'SI': 0.0},
        {'TI': 0.6, 'LI': 0.4, 'SI': 0.0},
    ]
}

result_df_2 = apply_weights_and_calculate(
    final_merged, combo_2['seasonal'], combo_2['non_seasonal']
)
orig_2, ci_2, count_2 = compute_summary(result_df_2)
print(f"Combination 2 → Original: {orig_2} m$  |  CI: {ci_2} m$  |  Count: {count_2}")

all_summary_rows.extend(
    build_summary_block(combo_2['label'], combo_2['seasonal'],
                        combo_2['non_seasonal'], orig_2, ci_2, count_2)
)

all_detail_frames['Combination 2'] = result_df_2[[
    'flightcost', 'final_adjustment_pct', 'Final Toggle',
    'Baseline cost', 'Original Toggle Amount', 'Toggle as per CI',
    'WT_TI', 'WT_LI', 'WT_SI', 'final_intensity',
    'TI Level', 'LI Level', 'SI_Toggle'
]].copy()

del result_df_2
print("✅ Combination 2 stored. Intermediate result cleared from memory.")

Combination 2 → Original: 185.03 m$  |  CI: 188.56 m$  |  Count: 45124
✅ Combination 2 stored. Intermediate result cleared from memory.


In [6]:
# ============================================================
# WRITE ALL TO EXCEL
# ============================================================
output_path = 'toggle_combinations_output.xlsx'
summary_df  = pd.DataFrame(all_summary_rows)

with pd.ExcelWriter(output_path, engine='openpyxl') as writer:

    # Summary sheet
    summary_df.to_excel(writer, sheet_name='Summary', index=False)

    # One sheet per combination
    for label, detail_df in all_detail_frames.items():
        detail_df.to_excel(writer, sheet_name=label[:31], index=False)

# ============================================================
# FORMAT SUMMARY SHEET
# ============================================================
wb = load_workbook(output_path)
ws = wb['Summary']

header_fill  = PatternFill("solid", fgColor="1F4E79")
header_font  = Font(bold=True, color="FFFFFF", size=10)
subhdr_fill  = PatternFill("solid", fgColor="2E75B6")
subhdr_font  = Font(bold=True, color="FFFFFF", size=10)
data_font    = Font(size=10)
center_align = Alignment(horizontal='center', vertical='center', wrap_text=True)
thin_border  = Border(
    left=Side(style='thin'), right=Side(style='thin'),
    top=Side(style='thin'),  bottom=Side(style='thin')
)

for row in ws.iter_rows():
    for cell in row:
        cell.font      = data_font
        cell.alignment = center_align
        cell.border    = thin_border
        if cell.row == 1:
            cell.fill = header_fill
            cell.font = header_font
        elif str(cell.value) in [
            'TI', 'SI', 'LI',
            'Original Toggle (m $)',
            'CI based toggle (m $)',
            'Count (CI toggle % < WU toggle %) for non NULL CI only'
        ]:
            cell.fill = subhdr_fill
            cell.font = subhdr_font

for col in ws.columns:
    max_len = max((len(str(cell.value)) for cell in col if cell.value), default=10)
    ws.column_dimensions[get_column_letter(col[0].column)].width = min(max_len + 4, 40)

wb.save(output_path)

print(f"✅ Excel written: {output_path}")
print(f"   Summary rows : {len(summary_df)}")
print(f"   Detail sheets: {list(all_detail_frames.keys())}")

✅ Excel written: toggle_combinations_output.xlsx
   Summary rows : 18
   Detail sheets: ['Combination 1', 'Combination 2']


In [7]:
# Run this in a new cell to diagnose
print("=== Combination 1 Non-Seasonal ===")
for row in all_detail_frames['Combination 1'][['WT_TI','WT_LI','WT_SI']].drop_duplicates().values:
    print(row)

print("\n=== Combination 2 Non-Seasonal ===")
for row in all_detail_frames['Combination 2'][['WT_TI','WT_LI','WT_SI']].drop_duplicates().values:
    print(row)

=== Combination 1 Non-Seasonal ===
[0.8 0.  0.2]
[0.6 0.4 0. ]
[0.5 0.  0.5]
[0.8 0.2 0. ]
[0.7 0.3 0. ]
[0.7 0.1 0.2]
[0.6 0.1 0.3]

=== Combination 2 Non-Seasonal ===
[0.8 0.  0.2]
[0.6 0.4 0. ]
[0.5 0.  0.5]
[0.8 0.2 0. ]
[0.7 0.3 0. ]
[0.7 0.1 0.2]
[0.6 0.1 0.3]


In [3]:
final_merged['flightrequestedAircraftCabinName'].unique()

array(['Super Midsize', 'Light', 'Midsize', nan, 'Turboprop',
       'Premium Light', 'Large', 'Premium Super-Mid'], dtype=object)

In [4]:
import pandas as pd
import numpy as np
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

# ============================================================
# AIRCRAFT CABIN MAPPING
# ============================================================
CABIN_MAP = {
    'PLJ'  : 'Premium Light',   # ← update to exact value in your data
    'PSMID': 'Premium Super-Mid'    # ← update to exact value in your data
}

# Verify cabin names in your data — run this to check:
# print(final_merged['flightrequestedAircraftCabinName'].unique())

# ============================================================
# HELPER: get_range_mean
# ============================================================
def get_range_mean(range_val):
    if pd.isna(range_val) or not isinstance(range_val, str):
        return np.nan
    try:
        if '+' in range_val:
            return float(range_val.replace('+', '').strip())
        parts = [float(x.strip()) for x in range_val.split('-')]
        return sum(parts) / len(parts)
    except Exception:
        return np.nan

# ============================================================
# HELPER: build_toggle_conditions
# ============================================================
def build_toggle_conditions(df):
    fi = df['final_intensity']
    conditions = [
        fi < 2,
        (fi >= 2) & (fi < 3),
        (fi >= 3) & (fi < 4),
        (fi >= 4) & (fi < 5),
        (fi >= 5) & (fi < 6),
        (fi >= 6) & (fi < 7),
        (fi >= 7) & (fi < 8),
        fi >= 8
    ]
    choices = [0.00, 0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40]
    return conditions, choices

# ============================================================
# CORE: Apply weights & compute all columns
# ============================================================
def apply_weights_and_calculate(df, seasonal_weights, non_seasonal_weights):
    df = df.copy()

    seasonal_mask     = df['SI_Toggle'].notna() & (df['SI_Toggle'] != 0)
    non_seasonal_mask = ~seasonal_mask
    bins = [0, 2, 3, 4, 5, 6, 10]

    def assign_tier_weights(mask, weight_list):
        if mask.sum() == 0:
            return
        tier_indices = np.digitize(df.loc[mask, 'final_intensity'], bins) - 1
        tier_indices = np.clip(5 - tier_indices, 0, 5)
        df.loc[mask, 'WT_TI'] = [weight_list[i]['TI'] for i in tier_indices]
        df.loc[mask, 'WT_SI'] = [weight_list[i]['SI'] for i in tier_indices]
        df.loc[mask, 'WT_LI'] = [weight_list[i]['LI'] for i in tier_indices]

    assign_tier_weights(seasonal_mask,     seasonal_weights)
    assign_tier_weights(non_seasonal_mask, non_seasonal_weights)

    # Recompute final_intensity
    df['final_intensity'] = (
        df['WT_TI'] * df['TI Level'].apply(get_range_mean) +
        df['WT_LI'] * df['LI Level'].apply(get_range_mean) +
        df['WT_SI'] * df['SI_Toggle'].fillna(0)
    )

    # Recompute Final Toggle
    conditions, choices = build_toggle_conditions(df)
    df['Final Toggle'] = np.select(conditions, choices, default=np.nan)

    # Clean invalid rows
    df = df[
        (df['final_adjustment_pct'] != -100) &
        (df['flightcost'].notna()) &
        (df['final_adjustment_pct'].notna()) &
        (abs(1 + df['final_adjustment_pct'] / 100) > 0.01)
    ].copy()

    # Derived columns
    divisor = 1 + df['final_adjustment_pct'] / 100
    df['Baseline cost']          = (df['flightcost'] / divisor) * 0.8
    df['Original Toggle Amount'] = (df['flightcost'] - df['flightcost'] / divisor) * 0.8
    df['Toggle as per CI']       = df['Baseline cost'] * df['Final Toggle']

    return df

# ============================================================
# SUMMARY: Compute metrics per cabin
# ============================================================
def compute_summary_by_cabin(df, cabin_col='flightrequestedAircraftCabinName'):
    results = {}
    for short_label, cabin_name in CABIN_MAP.items():
        cabin_df = df[df[cabin_col] == cabin_name]
        if cabin_df.empty:
            results[short_label] = (0.0, 0.0, 0)
            continue
        orig_m    = round(cabin_df['Original Toggle Amount'].sum() / 1_000_000, 2)
        ci_m      = round(cabin_df['Toggle as per CI'].sum()       / 1_000_000, 2)
        non_null  = cabin_df[cabin_df['Toggle as per CI'].notna()]
        count     = int((non_null['Toggle as per CI'] < non_null['Original Toggle Amount']).sum())
        results[short_label] = (orig_m, ci_m, count)
    return results

# ============================================================
# BUILD SUMMARY BLOCK (one block per cabin per combination)
# ============================================================
def build_summary_block(combo_label, seasonal_weights, non_seasonal_weights, cabin_metrics):
    all_rows = []

    for short_label, (orig_toggle, ci_toggle, count_ci_less) in cabin_metrics.items():

        # Cabin header row (e.g. PLJ / PSMID)
        all_rows.append({
            'Combination' : combo_label,
            'Cabin'       : short_label,
            'S_TI'        : 'TI',  'S_LI': 'LI',  'S_SI': 'SI',
            'NS_TI'       : 'TI',  'NS_LI': 'LI', 'NS_SI': 'SI',
            'Original Toggle (m $)'        : 'Original Toggle (m $)',
            'CI based toggle (m $)'        : 'CI based toggle (m $)',
            'Count (CI toggle < WU toggle)': 'Count (CI toggle % < WU toggle %) for non NULL CI only'
        })

        # 6 weight tier rows
        for i in range(6):
            sw  = seasonal_weights[i]
            nsw = non_seasonal_weights[i]
            all_rows.append({
                'Combination' : '',
                'Cabin'       : '',
                'S_TI'        : sw['TI'],   'S_LI': sw['LI'],   'S_SI': sw['SI'],
                'NS_TI'       : nsw['TI'],  'NS_LI': nsw['LI'], 'NS_SI': nsw['SI'],
                'Original Toggle (m $)'        : orig_toggle   if i == 0 else '',
                'CI based toggle (m $)'        : ci_toggle     if i == 0 else '',
                'Count (CI toggle < WU toggle)': count_ci_less if i == 0 else ''
            })

        # Blank separator
        all_rows.append({k: '' for k in all_rows[0].keys()})
        all_rows.append({k: '' for k in all_rows[0].keys()})

    return all_rows

# ============================================================
# STORAGE — reset every time Cell 1 is run
# ============================================================
all_summary_rows  = []
all_detail_frames = {}

print("✅ Helpers and storage initialized.")
print(f"   Cabin mapping  : {CABIN_MAP}")
print(f"   Unique cabins in data: {final_merged['flightrequestedAircraftCabinName'].unique()}")

✅ Helpers and storage initialized.
   Cabin mapping  : {'PLJ': 'Premium Light', 'PSMID': 'Premium Super-Mid'}
   Unique cabins in data: ['Super Midsize' 'Light' 'Midsize' nan 'Turboprop' 'Premium Light' 'Large'
 'Premium Super-Mid']


In [5]:
combo_1 = {
    'label': 'Combination 1',
    'seasonal': [
        {'TI': 0.6, 'SI': 0.3, 'LI': 0.1},
        {'TI': 0.7, 'SI': 0.2, 'LI': 0.1},
        {'TI': 0.8, 'SI': 0.2, 'LI': 0.0},
        {'TI': 0.8, 'SI': 0.2, 'LI': 0.0},
        {'TI': 0.8, 'SI': 0.2, 'LI': 0.0},
        {'TI': 0.5, 'SI': 0.5, 'LI': 0.0},
    ],
    'non_seasonal': [
        {'TI': 0.8, 'SI': 0.0, 'LI': 0.2},
        {'TI': 0.8, 'SI': 0.0, 'LI': 0.2},
        {'TI': 0.7, 'SI': 0.0, 'LI': 0.3},
        {'TI': 0.7, 'SI': 0.0, 'LI': 0.3},
        {'TI': 0.6, 'SI': 0.0, 'LI': 0.4},
        {'TI': 0.6, 'SI': 0.0, 'LI': 0.4},
    ]
}

# Run calculation on full dataset
result_df_1 = apply_weights_and_calculate(
    final_merged, combo_1['seasonal'], combo_1['non_seasonal']
)

# Compute metrics split by cabin
cabin_metrics_1 = compute_summary_by_cabin(result_df_1)
print(f"\n📊 Combination 1 Results:")
for cabin, (orig, ci, count) in cabin_metrics_1.items():
    print(f"   {cabin} → Original: {orig} m$  |  CI: {ci} m$  |  Count: {count}")

# Store summary block
all_summary_rows.extend(
    build_summary_block(
        combo_1['label'],
        combo_1['seasonal'],
        combo_1['non_seasonal'],
        cabin_metrics_1
    )
)

# Store full detail (all columns)
all_detail_frames['Combination 1'] = result_df_1.copy()

del result_df_1
print("\n✅ Combination 1 stored.")


📊 Combination 1 Results:
   PLJ → Original: 30.38 m$  |  CI: 31.91 m$  |  Count: 7836
   PSMID → Original: 3.33 m$  |  CI: 13.36 m$  |  Count: 219

✅ Combination 1 stored.


In [14]:
combo_2 = {
    'label': 'Combination 2',
    'seasonal': [
        {'TI': 0.6, 'SI': 0.3, 'LI': 0.1},
        {'TI': 0.7, 'SI': 0.2, 'LI': 0.1},
        {'TI': 0.8, 'SI': 0.2, 'LI': 0.0},
        {'TI': 0.8, 'SI': 0.2, 'LI': 0.0},
        {'TI': 0.8, 'SI': 0.2, 'LI': 0.0},
        {'TI': 0.5, 'SI': 0.5, 'LI': 0.0},
    ],
    'non_seasonal': [
        {'TI': 0.8, 'SI': 0.0, 'LI': 0.2},  # ← different weights
        {'TI': 0.8, 'SI': 0.0, 'LI': 0.2},
        {'TI': 0.7, 'SI': 0.0, 'LI': 0.3},
        {'TI': 0.7, 'SI': 0.0, 'LI': 0.3},
        {'TI': 0.6, 'SI': 0.0, 'LI': 0.4},
        {'TI': 0.6, 'SI': 0.0, 'LI': 0.4},
    ]
}

result_df_2 = apply_weights_and_calculate(
    final_merged, combo_2['seasonal'], combo_2['non_seasonal']
)

cabin_metrics_2 = compute_summary_by_cabin(result_df_2)
print(f"\n📊 Combination 2 Results:")
for cabin, (orig, ci, count) in cabin_metrics_2.items():
    print(f"   {cabin} → Original: {orig} m$  |  CI: {ci} m$  |  Count: {count}")

all_summary_rows.extend(
    build_summary_block(
        combo_2['label'],
        combo_2['seasonal'],
        combo_2['non_seasonal'],
        cabin_metrics_2
    )
)

all_detail_frames['Combination 2'] = result_df_2.copy()

del result_df_2
print("\n✅ Combination 2 stored.")


📊 Combination 2 Results:
   PLJ → Original: 30.38 m$  |  CI: 31.91 m$  |  Count: 7836
   PSMID → Original: 3.33 m$  |  CI: 13.36 m$  |  Count: 219

✅ Combination 2 stored.


In [ ]:
output_path = 'toggle_combinations_output.xlsx'
summary_df  = pd.DataFrame(all_summary_rows)

with pd.ExcelWriter(output_path, engine='openpyxl') as writer:

    # Sheet 1: Summary
    summary_df.to_excel(writer, sheet_name='Summary', index=False)

    # One sheet per combination — full dataset with all columns
    for label, detail_df in all_detail_frames.items():
        detail_df.to_excel(writer, sheet_name=label[:31], index=False)

# ============================================================
# FORMAT SUMMARY SHEET
# ============================================================
wb = load_workbook(output_path)
ws = wb['Summary']

combo_fill   = PatternFill("solid", fgColor="1F4E79")  # dark blue  — combination label
cabin_fill   = PatternFill("solid", fgColor="2E75B6")  # mid blue   — PLJ / PSMID header
subhdr_fill  = PatternFill("solid", fgColor="9DC3E6")  # light blue — TI/LI/SI row
combo_font   = Font(bold=True, color="FFFFFF", size=10)
cabin_font   = Font(bold=True, color="FFFFFF", size=10)
subhdr_font  = Font(bold=True, color="1F4E79", size=10)
data_font    = Font(size=10)
center_align = Alignment(horizontal='center', vertical='center', wrap_text=True)
thin_border  = Border(
    left=Side(style='thin'), right=Side(style='thin'),
    top=Side(style='thin'),  bottom=Side(style='thin')
)

for row in ws.iter_rows():
    for cell in row:
        cell.font      = data_font
        cell.alignment = center_align
        cell.border    = thin_border

        val = str(cell.value) if cell.value else ''

        # Column header row (row 1)
        if cell.row == 1:
            cell.fill = combo_fill
            cell.font = combo_font

        # Combination label rows
        elif val.startswith('Combination'):
            cell.fill = combo_fill
            cell.font = combo_font

        # Cabin label rows (PLJ / PSMID)
        elif val in list(CABIN_MAP.keys()):
            cell.fill = cabin_fill
            cell.font = cabin_font

        # Sub-header rows (TI / LI / SI labels)
        elif val in ['TI', 'LI', 'SI',
                     'Original Toggle (m $)',
                     'CI based toggle (m $)',
                     'Count (CI toggle % < WU toggle %) for non NULL CI only']:
            cell.fill = subhdr_fill
            cell.font = subhdr_font

# Auto column width
for col in ws.columns:
    max_len = max((len(str(cell.value)) for cell in col if cell.value), default=10)
    ws.column_dimensions[get_column_letter(col[0].column)].width = min(max_len + 4, 45)

wb.save(output_path)

print(f"✅ Excel written : {output_path}")
print(f"   Summary rows  : {len(summary_df)}")
print(f"   Detail sheets : {list(all_detail_frames.keys())}")
for label, df in all_detail_frames.items():
    print(f"   {label}: {len(df)} rows × {len(df.columns)} columns")

In [24]:
summary_df  = pd.DataFrame(all_summary_rows)

In [28]:
print(combo_1['label'], cabin_metrics_1)

Combination 1 {'PLJ': (np.float64(30.38), np.float64(31.91), 7836), 'PSMID': (np.float64(3.33), np.float64(13.36), 219)}


In [29]:
print(combo_2['label'], cabin_metrics_2)

Combination 2 {'PLJ': (np.float64(30.38), np.float64(31.91), 7836), 'PSMID': (np.float64(3.33), np.float64(13.36), 219)}
